## Reading the existing processed file for each dataset

In [6]:
import os
import copy
import sys
import argparse
import asyncio
from tqdm import tqdm
import random
from pprint import pprint
from dotenv import load_dotenv
from collections import defaultdict

# Ensure repository root is on sys.path so "Code" is importable when run directly
REPO_ROOT = '/dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
# Load environment variables from .env file
load_dotenv(os.path.join(REPO_ROOT, ".env"))

from Code.src.utils.io import read_json_file, save_json_file, get_paths

## Preparing the gold test data

In [9]:
def is_valid_sample(sample):
    """Check if a sample is valid (has non-null, non-empty raw-initial-ground-truth)."""
    arg_value = sample.get("raw-initial-ground-truth", None)
    if arg_value is None:
        return False
    if isinstance(arg_value, (list, str)) and len(arg_value) == 0:
        return False
    if isinstance(arg_value, list) and len(arg_value) == 1 and arg_value[0] == "null":
        return False
    return True


def filter_valid_samples(samples):
    """Filter out invalid samples (where raw-initial-ground-truth is None, empty, or "null")."""
    return [sample for sample in samples if is_valid_sample(sample)]


def get_unique_documents(samples, doc_id_key):
    """Get set of unique document IDs from samples."""
    return {sample.get(doc_id_key) for sample in samples if sample.get(doc_id_key) is not None}


def create_gold_test_set(processed_data, dataset_name, split_name, sample_count=500):
    """
    Create a gold set based on the split name by selecting samples only from that split.
    
    Args:
        processed_data: Dictionary with split keys ('train', 'dev', 'test')
        dataset_name: Name of the dataset ('DiscourseEE', 'PHEE', or 'CaseReportBench')
        split_name: Name of the split to use ('train', 'dev', or 'test')
        sample_count: Number of samples to select
    
    Returns:
        List of selected samples
    """
    # Determine document ID key
    doc_id_keys = {
        "DiscourseEE": "doc_id",
        "PHEE": "id",
        "CaseReportBench": "pmcid",
        "MACCROBAT": "doc_id"
    }
    doc_id_key = doc_id_keys.get(dataset_name)
    if doc_id_key is None:
        raise ValueError(f"Unknown dataset: {dataset_name}")
    
    # Get samples only from the specified split
    all_samples = processed_data.get(split_name, [])
    if not all_samples:
        raise ValueError(f"Split '{split_name}' not found in processed_data")
    
    print(f"\nTotal {split_name} samples available: {len(all_samples)}")
    
    # Filter valid samples
    valid_samples = filter_valid_samples(all_samples)
    invalid_count = len(all_samples) - len(valid_samples)
    print(f"Valid {split_name} samples: {len(valid_samples)}")
    print(f"Invalid {split_name} samples filtered out: {invalid_count}")
    
    # Calculate total unique documents
    total_docs = get_unique_documents(valid_samples, doc_id_key)
    print(f"Total unique documents in {split_name} split: {len(total_docs)}")
    
    # Randomly sample samples
    print(f"\nRandomly selecting {sample_count} samples from {split_name} split for gold set...")
    random.shuffle(valid_samples)
    selected_samples = valid_samples[:sample_count]
    
    # Calculate document coverage
    selected_doc_ids = get_unique_documents(selected_samples, doc_id_key)
    print(f"Selected {len(selected_samples)} samples from {len(selected_doc_ids)} unique documents (out of {len(total_docs)} total documents)")
    
    return selected_samples

In [10]:
# Set random seed for reproducibility
random.seed(42)
data_path = '/dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset'
# Process each dataset to create gold test sets
datasets_to_process = ["DiscourseEE", "PHEE", "CaseReportBench", "MACCROBAT"]

for dataset_name in datasets_to_process:
    print(f"\n{'='*60}")
    print(f"Creating gold test set for {dataset_name}")
    print(f"{'='*60}")
    
    # Read processed data
    dataset_dir = os.path.join(data_path, dataset_name)
    dataset_split_path = os.path.join(dataset_dir, f"{dataset_name}-final.json")
    
    if not os.path.exists(dataset_split_path):
        print(f"Warning: {dataset_split_path} not found. Skipping {dataset_name}.")
        continue
    
    processed_data = read_json_file(dataset_split_path)
    print(f"Loaded processed data from: {dataset_split_path}")
    
    # Create gold test set from test split
    split_name = 'test'
    test_samples = create_gold_test_set(processed_data, dataset_name, split_name, sample_count=500)
    
    # Save gold test set
    final_output_path = os.path.join(dataset_dir, f"{dataset_name}-gold-test.json")
    save_json_file({'test': test_samples}, final_output_path)
    
    print(f"\nGold test set saved to: {final_output_path}")
    print(f"Test samples: {len(test_samples)}")


Creating gold test set for DiscourseEE
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DiscourseEE/DiscourseEE-final.json
Loaded processed data from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DiscourseEE/DiscourseEE-final.json

Total test samples available: 1883
Valid test samples: 874
Invalid test samples filtered out: 1009
Total unique documents in test split: 99

Randomly selecting 500 samples from test split for gold set...
Selected 500 samples from 97 unique documents (out of 99 total documents)
Data saved to: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DiscourseEE/DiscourseEE-gold-test.json

Gold test set saved to: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/DiscourseEE/DiscourseEE-gold-test.json
Test samples: 500

Creating gold test set for PHEE
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/PHEE/PHEE-final.json
Loaded processed data from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/PHEE/

## Macrobat Dataset Merging

In [ ]:
# # Load the processed data and save MACCROBAT dataset
# split_names = ["train", "dev", "test"]
# dataset_name = "MACCROBAT"
# data_path = '/dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset'

# dataset_dir = os.path.join(data_path, dataset_name)
# final_dataset = {}

# for split_name in split_names:
#     dataset_split_path = os.path.join(dataset_dir, f"{dataset_name}-{split_name}.json")

#     if not os.path.exists(dataset_split_path):
#         print(f"Warning: {dataset_split_path} not found. Skipping {split_name}.")
#         continue

#     final_dataset[split_name] = read_json_file(dataset_split_path)
#     print(f"Loaded {split_name}: {len(final_dataset[split_name])} samples")

# output_path = os.path.join(dataset_dir, f"{dataset_name}-final.json")
# save_json_file(final_dataset, output_path)
# print(f"\nMACCROBAT dataset saved to: {output_path}")
# print(f"Train: {len(final_dataset.get('train', []))} | Dev: {len(final_dataset.get('dev', []))} | Test: {len(final_dataset.get('test', []))}")

Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/MACCROBAT/MACCROBAT-train.json
Loaded train: 5508 samples
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/MACCROBAT/MACCROBAT-dev.json
Loaded dev: 671 samples
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/MACCROBAT/MACCROBAT-test.json
Loaded test: 741 samples
Data saved to: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/MACCROBAT/MACCROBAT-final.json

MACCROBAT dataset saved to: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/MACCROBAT/MACCROBAT-final.json
Train: 5508 | Dev: 671 | Test: 741


## CaseReportBench Spit Creation

In [26]:
# def create_splits_casereportbench(processed_data, test_count=500):
#     """
#     Create new splits for CaseReportBench dataset.
    
#     Args:
#         processed_data: Dictionary with 'test' key (all data)
#         test_count: Number of samples for test set
    
#     Returns:
#         Dictionary with new 'train', 'dev', 'test' splits
#     """
#     doc_id_key = "pmcid"
    
#     # Get all samples
#     all_samples = processed_data.get('test', [])
#     print(f"\nTotal samples available: {len(all_samples)}")
    
#     # Filter valid samples only
#     valid_samples = filter_valid_samples(all_samples)
#     invalid_count = len(all_samples) - len(valid_samples)
#     print(f"Valid samples: {len(valid_samples)}")
#     print(f"Invalid samples filtered out: {invalid_count}")
    
#     # Calculate total unique documents
#     total_docs = set(sample.get(doc_id_key) for sample in valid_samples if sample.get(doc_id_key) is not None)
#     print(f"Total unique documents: {len(total_docs)}")
    
#     # Shuffle all valid samples
#     random.shuffle(valid_samples)
    
#     # Step 1: Randomly select test set (500 samples)
#     print(f"\nRandomly selecting {test_count} samples for test set...")
#     test_samples = valid_samples[:test_count]
#     remaining_samples = valid_samples[test_count:]
    
#     # Calculate document coverage for test
#     test_doc_ids = set(sample.get(doc_id_key) for sample in test_samples if sample.get(doc_id_key) is not None)
#     print(f"Test set: {len(test_samples)} samples from {len(test_doc_ids)} unique documents (out of {len(total_docs)} total documents)")
    
#     # Step 2: Remaining data split into train (80%) and dev (20%)
#     print(f"\nRemaining samples: {len(remaining_samples)}")
    
#     # Split: 80% train, 20% dev
#     train_size = int(len(remaining_samples) * 0.8)
#     train_samples = remaining_samples[:train_size]
#     dev_samples = remaining_samples[train_size:]
    
#     # Calculate unique documents covered for train and dev
#     train_doc_ids = set(sample.get(doc_id_key) for sample in train_samples if sample.get(doc_id_key) is not None)
#     dev_doc_ids = set(sample.get(doc_id_key) for sample in dev_samples if sample.get(doc_id_key) is not None)
    
#     print(f"Train set: {len(train_samples)} samples from {len(train_doc_ids)} unique documents (out of {len(total_docs)} total documents)")
#     print(f"Dev set: {len(dev_samples)} samples from {len(dev_doc_ids)} unique documents (out of {len(total_docs)} total documents)")
    
#     # Update serial numbers
#     for idx, sample in enumerate(test_samples, 1):
#         sample['serial-number'] = f"test-{idx}"
#     for idx, sample in enumerate(dev_samples, 1):
#         sample['serial-number'] = f"dev-{idx}"
#     for idx, sample in enumerate(train_samples, 1):
#         sample['serial-number'] = f"train-{idx}"
    
#     return {
#         'train': train_samples,
#         'dev': dev_samples,
#         'test': test_samples
#     }

# # Set random seed for reproducibility
# random.seed(42)
# data_path = '/dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset'
# # Process each dataset to create gold test sets
# datasets_to_process = ["CaseReportBench"]

# for dataset_name in datasets_to_process:
#     print(f"\n{'='*60}")
#     print(f"Creating splits for {dataset_name}")
#     print(f"{'='*60}")
    
#     # Read processed data
#     dataset_dir = os.path.join(data_path, dataset_name)
#     dataset_split_path = os.path.join(dataset_dir, f"{dataset_name}-processed.json")
    
#     if not os.path.exists(dataset_split_path):
#         print(f"Warning: {dataset_split_path} not found. Skipping {dataset_name}.")
#         continue
    
#     processed_data = read_json_file(dataset_split_path)
#     print(f"Loaded processed data from: {dataset_split_path}")
    
#     # Create gold test set from test split
#     new_splits = create_splits_casereportbench(processed_data, test_count=500)
    
#     # Save gold test set
#     final_output_path = os.path.join(dataset_dir, f"{dataset_name}-final.json")
#     save_json_file({'test': test_samples}, final_output_path)
    
#     print(f"Test samples: {len(test_samples)}")


Creating splits for CaseReportBench
Reading JSON file from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/CaseReportBench/CaseReportBench-processed.json
Loaded processed data from: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/CaseReportBench/CaseReportBench-processed.json

Total samples available: 2484
Valid samples: 1320
Invalid samples filtered out: 1164
Total unique documents: 138

Randomly selecting 500 samples for test set...
Test set: 500 samples from 138 unique documents (out of 138 total documents)

Remaining samples: 820
Train set: 656 samples from 138 unique documents (out of 138 total documents)
Dev set: 164 samples from 97 unique documents (out of 138 total documents)
Data saved to: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/CaseReportBench/CaseReportBench-final.json

Gold test set saved to: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/CaseReportBench/CaseReportBench-final.json
Test samples: 500


## This will be useful in the future for creating train and dev samples 

In [ ]:
# def is_valid_sample(sample):
#     """
#     Check if a sample is valid (has non-null, non-empty raw-initial-ground-truth).
    
#     Args:
#         sample: Sample dictionary
    
#     Returns:
#         True if sample is valid, False otherwise
#     """
#     arg_value = sample.get("raw-initial-ground-truth", None)
#     if arg_value is None:
#         return False
#     if isinstance(arg_value, (list, str)) and len(arg_value) == 0:
#         return False
#     if isinstance(arg_value, list) and len(arg_value) == 1 and arg_value[0] == "null":
#         return False
#     return True


# def filter_valid_samples(samples):
#     """
#     Filter out invalid samples (where raw-initial-ground-truth is None, empty, or "null").
    
#     Args:
#         samples: List of sample dictionaries
    
#     Returns:
#         List of valid samples
#     """
#     return [sample for sample in samples if is_valid_sample(sample)]


# def group_samples_by_document(samples, doc_id_key):
#     """
#     Group samples by document ID to maximize document coverage.
#     Only groups valid samples (filters out invalid ones).
    
#     Args:
#         samples: List of sample dictionaries
#         doc_id_key: Key to use for document identification ('doc_id', 'id', or 'pmcid')
    
#     Returns:
#         Dictionary mapping document IDs to lists of valid samples
#     """
#     doc_groups = defaultdict(list)
#     for sample in samples:
#         # Only group valid samples
#         if is_valid_sample(sample):
#             doc_id = sample.get(doc_id_key)
#             if doc_id is not None:
#                 doc_groups[doc_id].append(sample)
#     return doc_groups


# def select_samples_maximizing_document_coverage(doc_groups, target_count):
#     """
#     Select samples to maximize the number of unique documents covered.
    
#     Strategy:
#     1. Sort documents by number of samples (ascending) to prioritize documents with fewer samples
#     2. Iteratively add documents until we reach the target count
#     3. If we exceed target, randomly sample from the last document added
    
#     Args:
#         doc_groups: Dictionary mapping document IDs to lists of samples
#         target_count: Target number of samples to select
    
#     Returns:
#         Tuple of (selected_samples, selected_doc_ids, remaining_doc_groups)
#         where remaining_doc_groups has the selected samples removed
#     """
#     # Create a deep copy to avoid modifying original
#     remaining_doc_groups = {k: v.copy() for k, v in doc_groups.items()}
    
#     # Sort documents by number of samples (ascending)
#     sorted_docs = sorted(remaining_doc_groups.items(), key=lambda x: len(x[1]))
    
#     selected_samples = []
#     selected_doc_ids = set()
    
#     # First pass: add entire documents until we approach target
#     for doc_id, samples in sorted_docs:
#         if len(selected_samples) + len(samples) <= target_count:
#             selected_samples.extend(samples)
#             selected_doc_ids.add(doc_id)
#             # Remove all samples from this document
#             remaining_doc_groups[doc_id] = []
#         elif len(selected_samples) < target_count:
#             # We need some samples from this document but not all
#             remaining_needed = target_count - len(selected_samples)
#             # Randomly sample from this document (shuffle a copy to avoid modifying original)
#             samples_copy = samples.copy()
#             random.shuffle(samples_copy)
#             selected_samples.extend(samples_copy[:remaining_needed])
#             selected_doc_ids.add(doc_id)
#             # Remove selected samples from remaining
#             remaining_doc_groups[doc_id] = samples_copy[remaining_needed:]
#             break
    
#     # If we haven't reached target, add more documents
#     if len(selected_samples) < target_count:
#         sorted_docs = sorted(remaining_doc_groups.items(), key=lambda x: len(x[1]) if x[1] else 0)
#         for doc_id, samples in sorted_docs:
#             if not samples:  # Skip empty documents
#                 continue
#             if doc_id not in selected_doc_ids:
#                 remaining_needed = target_count - len(selected_samples)
#                 if remaining_needed <= 0:
#                     break
#                 # Add samples from this document (shuffle a copy to avoid modifying original)
#                 samples_copy = samples.copy()
#                 random.shuffle(samples_copy)
#                 num_to_take = min(remaining_needed, len(samples_copy))
#                 selected_samples.extend(samples_copy[:num_to_take])
#                 selected_doc_ids.add(doc_id)
#                 # Remove selected samples from remaining
#                 remaining_doc_groups[doc_id] = samples_copy[num_to_take:]
#                 if len(selected_samples) >= target_count:
#                     break
    
#     # Trim to exact target if we exceeded
#     if len(selected_samples) > target_count:
#         selected_samples = selected_samples[:target_count]
    
#     # Final validation: ensure all selected samples are valid
#     selected_samples = [s for s in selected_samples if is_valid_sample(s)]
    
#     # Clean up empty documents from remaining
#     remaining_doc_groups = {k: v for k, v in remaining_doc_groups.items() if v}
    
#     return selected_samples, selected_doc_ids, remaining_doc_groups


# def create_splits_discourseee_phee(processed_data, dataset_name, test_count=500, train_count=1500, dev_count=500):
#     """
#     Create new splits for DiscourseEE or PHEE datasets.
    
#     Args:
#         processed_data: Dictionary with 'train', 'dev', 'test' keys
#         dataset_name: Name of the dataset ('DiscourseEE' or 'PHEE')
#         test_count: Number of samples for test set
#         train_count: Number of samples for train set
#         dev_count: Number of samples for dev set
    
#     Returns:
#         Dictionary with new 'train', 'dev', 'test' splits
#     """
#     # Determine document ID key
#     if dataset_name == "DiscourseEE":
#         doc_id_key = "doc_id"
#     elif dataset_name == "PHEE":
#         doc_id_key = "id"
#     else:
#         raise ValueError(f"Unknown dataset: {dataset_name}")
    
#     # Process each split independently
#     new_splits = {}
    
#     # Process train split
#     if 'train' in processed_data:
#         train_samples_raw = processed_data['train']
#         print(f"\n{'='*60}")
#         print(f"Processing TRAIN split")
#         print(f"{'='*60}")
#         print(f"Total train samples available: {len(train_samples_raw)}")
        
#         # Filter valid samples
#         valid_train = filter_valid_samples(train_samples_raw)
#         invalid_train = len(train_samples_raw) - len(valid_train)
#         print(f"Valid train samples: {len(valid_train)}")
#         print(f"Invalid train samples filtered out: {invalid_train}")
        
#         # Calculate total unique documents in train split
#         total_train_docs = set(sample.get(doc_id_key) for sample in valid_train if sample.get(doc_id_key) is not None)
#         print(f"Total unique documents in train split: {len(total_train_docs)}")
        
#         # Randomly sample train samples
#         print(f"\nRandomly selecting {train_count} samples from TRAIN split...")
#         random.shuffle(valid_train)
#         train_samples = valid_train[:train_count]
        
#         # Calculate document coverage
#         train_doc_ids = set(sample.get(doc_id_key) for sample in train_samples if sample.get(doc_id_key) is not None)
#         print(f"Selected {len(train_samples)} train samples from {len(train_doc_ids)} unique documents (out of {len(total_train_docs)} total documents)")
#         new_splits['train'] = train_samples
#     else:
#         new_splits['train'] = []
    
#     # Process dev split
#     if 'dev' in processed_data:
#         dev_samples_raw = processed_data['dev']
#         print(f"\n{'='*60}")
#         print(f"Processing DEV split")
#         print(f"{'='*60}")
#         print(f"Total dev samples available: {len(dev_samples_raw)}")
        
#         # Filter valid samples
#         valid_dev = filter_valid_samples(dev_samples_raw)
#         invalid_dev = len(dev_samples_raw) - len(valid_dev)
#         print(f"Valid dev samples: {len(valid_dev)}")
#         print(f"Invalid dev samples filtered out: {invalid_dev}")
        
#         # Calculate total unique documents in dev split
#         total_dev_docs = set(sample.get(doc_id_key) for sample in valid_dev if sample.get(doc_id_key) is not None)
#         print(f"Total unique documents in dev split: {len(total_dev_docs)}")
        
#         # Randomly sample dev samples
#         print(f"\nRandomly selecting {dev_count} samples from DEV split...")
#         random.shuffle(valid_dev)
#         dev_samples = valid_dev[:dev_count]
        
#         # Calculate document coverage
#         dev_doc_ids = set(sample.get(doc_id_key) for sample in dev_samples if sample.get(doc_id_key) is not None)
#         print(f"Selected {len(dev_samples)} dev samples from {len(dev_doc_ids)} unique documents (out of {len(total_dev_docs)} total documents)")
#         new_splits['dev'] = dev_samples
#     else:
#         new_splits['dev'] = []
    
#     # Process test split
#     if 'test' in processed_data:
#         test_samples_raw = processed_data['test']
#         print(f"\n{'='*60}")
#         print(f"Processing TEST split")
#         print(f"{'='*60}")
#         print(f"Total test samples available: {len(test_samples_raw)}")
        
#         # Filter valid samples
#         valid_test = filter_valid_samples(test_samples_raw)
#         invalid_test = len(test_samples_raw) - len(valid_test)
#         print(f"Valid test samples: {len(valid_test)}")
#         print(f"Invalid test samples filtered out: {invalid_test}")
        
#         # Calculate total unique documents in test split
#         total_test_docs = set(sample.get(doc_id_key) for sample in valid_test if sample.get(doc_id_key) is not None)
#         print(f"Total unique documents in test split: {len(total_test_docs)}")
        
#         # Randomly sample test samples
#         print(f"\nRandomly selecting {test_count} samples from TEST split...")
#         random.shuffle(valid_test)
#         test_samples = valid_test[:test_count]
        
#         # Calculate document coverage
#         test_doc_ids = set(sample.get(doc_id_key) for sample in test_samples if sample.get(doc_id_key) is not None)
#         print(f"Selected {len(test_samples)} test samples from {len(test_doc_ids)} unique documents (out of {len(total_test_docs)} total documents)")
#         new_splits['test'] = test_samples
#     else:
#         new_splits['test'] = []
    
#     # Update serial numbers
#     for idx, sample in enumerate(new_splits.get('test', []), 1):
#         sample['serial-number'] = f"test-{idx}"
#     for idx, sample in enumerate(new_splits.get('dev', []), 1):
#         sample['serial-number'] = f"dev-{idx}"
#     for idx, sample in enumerate(new_splits.get('train', []), 1):
#         sample['serial-number'] = f"train-{idx}"
    
#     return new_splits


# def create_splits_casereportbench(processed_data, test_count=500):
#     """
#     Create new splits for CaseReportBench dataset.
    
#     Args:
#         processed_data: Dictionary with 'test' key (all data)
#         test_count: Number of samples for test set
    
#     Returns:
#         Dictionary with new 'train', 'dev', 'test' splits
#     """
#     doc_id_key = "pmcid"
    
#     # Get all samples
#     all_samples = processed_data.get('test', [])
#     print(f"\nTotal samples available: {len(all_samples)}")
    
#     # Filter valid samples only
#     valid_samples = filter_valid_samples(all_samples)
#     invalid_count = len(all_samples) - len(valid_samples)
#     print(f"Valid samples: {len(valid_samples)}")
#     print(f"Invalid samples filtered out: {invalid_count}")
    
#     # Calculate total unique documents
#     total_docs = set(sample.get(doc_id_key) for sample in valid_samples if sample.get(doc_id_key) is not None)
#     print(f"Total unique documents: {len(total_docs)}")
    
#     # Shuffle all valid samples
#     random.shuffle(valid_samples)
    
#     # Step 1: Randomly select test set (500 samples)
#     print(f"\nRandomly selecting {test_count} samples for test set...")
#     test_samples = valid_samples[:test_count]
#     remaining_samples = valid_samples[test_count:]
    
#     # Calculate document coverage for test
#     test_doc_ids = set(sample.get(doc_id_key) for sample in test_samples if sample.get(doc_id_key) is not None)
#     print(f"Test set: {len(test_samples)} samples from {len(test_doc_ids)} unique documents (out of {len(total_docs)} total documents)")
    
#     # Step 2: Remaining data split into train (80%) and dev (20%)
#     print(f"\nRemaining samples: {len(remaining_samples)}")
    
#     # Split: 80% train, 20% dev
#     train_size = int(len(remaining_samples) * 0.8)
#     train_samples = remaining_samples[:train_size]
#     dev_samples = remaining_samples[train_size:]
    
#     # Calculate unique documents covered for train and dev
#     train_doc_ids = set(sample.get(doc_id_key) for sample in train_samples if sample.get(doc_id_key) is not None)
#     dev_doc_ids = set(sample.get(doc_id_key) for sample in dev_samples if sample.get(doc_id_key) is not None)
    
#     print(f"Train set: {len(train_samples)} samples from {len(train_doc_ids)} unique documents (out of {len(total_docs)} total documents)")
#     print(f"Dev set: {len(dev_samples)} samples from {len(dev_doc_ids)} unique documents (out of {len(total_docs)} total documents)")
    
#     # Update serial numbers
#     for idx, sample in enumerate(test_samples, 1):
#         sample['serial-number'] = f"test-{idx}"
#     for idx, sample in enumerate(dev_samples, 1):
#         sample['serial-number'] = f"dev-{idx}"
#     for idx, sample in enumerate(train_samples, 1):
#         sample['serial-number'] = f"train-{idx}"
    
#     return {
#         'train': train_samples,
#         'dev': dev_samples,
#         'test': test_samples
#     }